**UTS Pembelajaran Mesin – Klasifikasi round_winner Counter-Strike 2
Algoritma: **Decision Tree** dan K-Nearest Neighbors (KNN)**



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

DATA_PATH = "03_counter_strike_2.csv"
OUT_DIR = "fig"; os.makedirs(OUT_DIR, exist_ok=True)
RS = 42
sns.set_theme(style="whitegrid")

def save(name):
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{name}.png", dpi=150); plt.close()

## 1. LOAD DATA & PEMAHAMAN DATA

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data :", df.shape)
print(df.head())
df.info()
print(df.describe().T)
print("Missing value:\n", df.isnull().sum())
print("Data duplikat:", df.duplicated().sum())

num_cols = df.select_dtypes(include="number").columns.tolist()

## 2. EDA

In [ ]:
plt.figure(figsize=(5, 4))
ax = sns.countplot(x="round_winner", hue="round_winner", data=df, palette=["#4C72B0", "#DD8452"], legend=False)
for c in ax.containers: ax.bar_label(c)
plt.title("Distribusi Kelas round_winner"); save("01_target")
print(df["round_winner"].value_counts(normalize=True))

df[num_cols].hist(bins=25, figsize=(14, 10), color="#4C72B0", edgecolor="white")
plt.suptitle("Distribusi Fitur Numerik"); save("02_histogram")

fig, axs = plt.subplots(3, 4, figsize=(15, 9))
for a, c in zip(axs.ravel(), num_cols):
    sns.boxplot(y=df[c], ax=a, color="#8FB9E0"); a.set_title(c)
save("03_boxplot")
for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75]); i = q3 - q1
    n = ((df[c] < q1 - 1.5 * i) | (df[c] > q3 + 1.5 * i)).sum()
    print(f"Outlier {c}: {n}")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(y="map", data=df, ax=axs[0], order=df["map"].value_counts().index, color="#4C72B0")
axs[0].set_title("Jumlah Data per Map")
sns.countplot(x="bomb_planted", hue="round_winner", data=df, ax=axs[1], palette=["#4C72B0", "#DD8452"])
axs[1].set_title("bomb_planted vs round_winner")
sns.countplot(y="map", hue="round_winner", data=df, ax=axs[2], palette=["#4C72B0", "#DD8452"])
axs[2].set_title("Map vs round_winner"); save("04_kategorikal")

tmp = df.copy()
tmp["bomb_planted"] = tmp["bomb_planted"].astype(int)
tmp["winner_T"] = (tmp["round_winner"] == "T").astype(int)
corr = tmp.drop(columns=["map", "round_winner"]).corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 8})
plt.title("Matriks Korelasi (winner_T: T=1, CT=0)"); save("05_korelasi")
print(corr["winner_T"].sort_values(ascending=False))

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for a, cols, t in zip(axs, [["ct_health", "t_health"], ["ct_armor", "t_armor"], ["ct_money", "t_money"]],
                      ["Health", "Armor", "Money"]):
    df.groupby("round_winner")[cols].mean().plot(kind="bar", ax=a, rot=0)
    a.set_title(f"Rata-rata {t} per pemenang ronde")
save("06_rata_kelas")

## 3. PREPROCESSING

In [ ]:
data = df.drop_duplicates().copy()
data["bomb_planted"] = data["bomb_planted"].astype(int)
data = pd.get_dummies(data, columns=["map"], drop_first=False, dtype=int)
le = LabelEncoder()
y = le.fit_transform(data["round_winner"])
print("Label:", dict(zip(le.classes_, le.transform(le.classes_))))
X = data.drop(columns=["round_winner"])
print("Fitur akhir:", X.shape[1], list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 4. FUNGSI EVALUASI

In [ ]:
results = {}
def evaluate(name, model, Xtr, Xte, ytr=y_train, yte=y_test):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    r = dict(Akurasi_Train=accuracy_score(ytr, model.predict(Xtr)),
             Akurasi_Test=accuracy_score(yte, pred),
             Precision=precision_score(yte, pred), Recall=recall_score(yte, pred),
             F1=f1_score(yte, pred))
    results[name] = r
    print(f"\n=== {name} ===")
    print({k: round(v, 4) for k, v in r.items()})
    print(classification_report(yte, pred, target_names=le.classes_))
    return model, pred

def cm_plot(pred, title, fname):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(cmap="Blues", values_format="d")
    plt.title(title); save(fname); return cm

## 5. DECISION TREE

In [ ]:
dt_base, pred_dt_base = evaluate("Decision Tree (default)",
                                 DecisionTreeClassifier(random_state=RS), X_train, X_test)
cm_dt_base = cm_plot(pred_dt_base, "CM Decision Tree (default)", "07_cm_dt_default")

depths = range(1, 21); tr_acc, te_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RS).fit(X_train, y_train)
    tr_acc.append(m.score(X_train, y_train)); te_acc.append(m.score(X_test, y_test))
plt.figure(figsize=(7, 4))
plt.plot(depths, tr_acc, marker="o", label="Train"); plt.plot(depths, te_acc, marker="o", label="Test")
plt.xlabel("max_depth"); plt.ylabel("Akurasi"); plt.title("Pengaruh max_depth pada Decision Tree")
plt.legend(); save("08_dt_depth")

param_dt = {"max_depth": [3, 5, 7, 10, 15, None],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "criterion": ["gini", "entropy"]}
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
gs_dt = GridSearchCV(DecisionTreeClassifier(random_state=RS), param_dt, cv=cv,
                     scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
print("Best param DT:", gs_dt.best_params_, "| CV acc:", round(gs_dt.best_score_, 4))
dt_best, pred_dt_best = evaluate("Decision Tree (tuned)", gs_dt.best_estimator_, X_train, X_test)
cm_dt_best = cm_plot(pred_dt_best, "CM Decision Tree (tuned)", "09_cm_dt_tuned")

imp = pd.Series(dt_best.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10))
plt.figure(figsize=(7, 6)); imp.head(12)[::-1].plot(kind="barh", color="#4C72B0")
plt.title("Feature Importance - Decision Tree (tuned)"); save("10_dt_importance")

plt.figure(figsize=(15, 7))
plot_tree(dt_best, max_depth=2, feature_names=X.columns, class_names=le.classes_,
          filled=True, rounded=True, fontsize=11)
plt.title("Visualisasi Decision Tree (2 level teratas)"); save("11_dt_tree")

## 6. KNN

In [ ]:
knn_raw, _ = evaluate("KNN (k=5, tanpa scaling)", KNeighborsClassifier(n_neighbors=5), X_train, X_test)
knn_base, pred_knn_base = evaluate("KNN (k=5, scaling)", KNeighborsClassifier(n_neighbors=5),
                                   X_train_s, X_test_s)
cm_knn_base = cm_plot(pred_knn_base, "CM KNN (k=5, scaling)", "12_cm_knn_default")

ks = range(1, 31); k_tr, k_te, k_cv = [], [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    k_tr.append(m.score(X_train_s, y_train)); k_te.append(m.score(X_test_s, y_test))
    k_cv.append(cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=cv).mean())
plt.figure(figsize=(7, 4))
plt.plot(ks, k_tr, label="Train"); plt.plot(ks, k_cv, label="CV (5-fold)"); plt.plot(ks, k_te, label="Test")
plt.xlabel("Nilai k"); plt.ylabel("Akurasi"); plt.title("Pengaruh Nilai k pada KNN"); plt.legend()
save("13_knn_k")
print("k terbaik (CV):", ks[int(np.argmax(k_cv))], round(max(k_cv), 4))

param_knn = {"n_neighbors": list(range(1, 31)), "weights": ["uniform", "distance"],
             "metric": ["euclidean", "manhattan"]}
gs_knn = GridSearchCV(KNeighborsClassifier(), param_knn, cv=cv, scoring="accuracy",
                      n_jobs=-1).fit(X_train_s, y_train)
print("Best param KNN:", gs_knn.best_params_, "| CV acc:", round(gs_knn.best_score_, 4))
knn_best, pred_knn_best = evaluate("KNN (tuned)", gs_knn.best_estimator_, X_train_s, X_test_s)
cm_knn_best = cm_plot(pred_knn_best, "CM KNN (tuned)", "14_cm_knn_tuned")

## 7. PERBANDINGAN & CROSS VALIDATION

In [ ]:
res = pd.DataFrame(results).T.round(4)
print("\n", res)
res.to_csv("hasil_model.csv")

res[["Akurasi_Test", "Precision", "Recall", "F1"]].plot(kind="bar", figsize=(11, 5), rot=20)
plt.ylim(0.5, 1); plt.title("Perbandingan Performa Model"); plt.legend(loc="lower right")
save("15_perbandingan")

cv_dt = cross_val_score(dt_best, X, y, cv=cv)
cv_knn = cross_val_score(Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier(**{
    k.replace("n_neighbors", "n_neighbors"): v for k, v in gs_knn.best_params_.items()}))]), X, y, cv=cv)
print("CV 5-fold DT  :", cv_dt.round(4), cv_dt.mean().round(4))
print("CV 5-fold KNN :", cv_knn.round(4), cv_knn.mean().round(4))

print("\nConfusion matrix DT tuned:\n", cm_dt_best)
print("Confusion matrix KNN tuned:\n", cm_knn_best)